In [13]:
import json
from esg_lib.analysis_api import build_esg_analysis

# 1. Paramètres "fake" juste pour la simulation
AWS_ACCOUNT_ID = "123456789012"
DATASET_ARN = "arn:aws:quicksight:eu-west-1:123456789012:dataset/ESG_dataset"
DATASET_ID = "ESG_dataset"

# 2. Mappings colonnes -> champs ESG attendus par la lib
# (le dataset réel n'est pas chargé)
MAPPINGS = {
    "sector": "Sector",
    "subsector": "SubSector",
    "country": "Country",
    "date": "Year",
    "year": "Year",  
    "emissions_total": "CO2_Emissions",
    "carbon_intensity": "CO2_Intensity",
    "intensity_target": "Target_Intensity",
}

# Construire l'objet "analysis" 
analysis_json = build_esg_analysis(
    aws_account_id=AWS_ACCOUNT_ID,
    dataset_arn=DATASET_ARN,
    dataset_id=DATASET_ID,
    mappings=MAPPINGS,
)

# L’enregistrer dans un fichier JSON local
out_file = "esg_dashboard_output_local.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(analysis_json, f, indent=2)

print(f" Fichier JSON généré : {out_file}")


 Fichier JSON généré : esg_dashboard_output_local.json


In [ ]:
import json
from pathlib import Path

def generate_html_dashboard(json_file, out_file="dashboard_preview.html"):
    # Charger le JSON
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    sheets = data["Definition"]["Sheets"]

    # On génère un HTML très simple : un bloc par sheet, un bloc par visual
    html_parts = []
    html_parts.append("""
    <html>
    <head>
        <meta charset="utf-8">
        <style>
            body { font-family: Arial, sans-serif; margin: 20px; }
            .sheet { border: 1px solid #ccc; margin-bottom: 30px; padding: 15px; border-radius: 6px; }
            .sheet h2 { margin-top: 0; }
            .visual { border: 1px solid #ddd; padding: 8px; margin: 5px 0; border-radius: 4px; }
            .grid-info { font-size: 11px; color: #666; }
        </style>
    </head>
    <body>
    <h1>ESG Dashboard – Aperçu local</h1>
    """)

    for sheet in sheets:
        title = sheet.get("Name", sheet.get("Title", "Sheet"))
        html_parts.append(f'<div class="sheet"><h2>{title}</h2>')

        layout = sheet.get("Layout", {}).get("Configuration", {}).get("GridLayout", {})
        elements = {el["ElementId"]: el for el in layout.get("Elements", [])}

        visuals = sheet.get("Visuals", [])
        for vis in visuals:
            v_id = vis.get("VisualId", "no-id")
            v_title = vis.get("Title", {}).get("Visibility", "Visual")
            
            if isinstance(v_title, dict):
                v_title = v_title.get("PlainText", "Visual")

            grid = elements.get(v_id, {})
            grid_info = f'row={grid.get("RowIndex", "?")}, col={grid.get("ColumnIndex", "?")}, ' \
                        f'rowSpan={grid.get("RowSpan", "?")}, colSpan={grid.get("ColumnSpan", "?")}'

            html_parts.append(f'<div class="visual"><strong>{v_title}</strong>'
                              f'<div class="grid-info">{v_id} – {grid_info}</div></div>')

        html_parts.append("</div>")  # fin sheet

    html_parts.append("</body></html>")

    html_str = "\n".join(html_parts)
    Path(out_file).write_text(html_str, encoding="utf-8")
    print(f" Fichier HTML généré : {out_file}")

    return out_file


In [19]:
html_file = generate_html_dashboard("esg_dashboard_output_local.json",
                                    out_file="dashboard_preview.html")
html_file

 Fichier HTML généré : dashboard_preview.html


'dashboard_preview.html'